## 고객 세그멘테이션 및 RFM 분석 

In [80]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')


### 0. 데이터 준비

#### column info
1. 고객 정보 (Demographic)
- ID, Year_Birth, Education, Marital_Status, Income, Kidhome, Teenhome
- → 고객의 인구통계학적 특성
- 활용) 타겟 마케팅/세분화(Segmentation)

2. 가입/등록 시점
- Dt_Customer: 고객이 등록한 날짜 
- 활용) 고객 생애 가치(LTV) 추정

3. 고객 행동/구매 정보
- Recency: 최근 구매 이후 경과일 
- 활용) RFM 분석의 R

- MntWines, MntFruits, MntMeatProducts, MntFishProducts, MntSweetProducts, MntGoldProds:
- → 상품군별 지출 금액. RFM 분석의 M
- 활용) 제품 선호도/고객 가치 분석

4. 구매 채널 정보
- NumDealsPurchases: 할인 딜 이용 횟수

- NumWebPurchases, NumCatalogPurchases, NumStorePurchases
- → 구매 채널별 구매 건수 
- 활용) Omni-channel 분석에 활용

5. 웹 행동 데이터
- NumWebVisitsMonth: 월간 웹사이트 방문 횟수
- 활용) 웹 채널의 관여도(engagement) 추정

6. 마케팅 캠페인 반응
- AcceptedCmp1 ~ AcceptedCmp5, Response
- → 마케팅 캠페인에 대한 수용/반응 이력

7. 불만 및 수동 변수
- Complain: 불만 여부
- Z_CostContact, Z_Revenue: 일반화된 더미 변수 

In [81]:

path = '../datasets/ml/crm/marketing_campaign.csv'
df = pd.read_csv(path, sep='\t')
df.head()


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [83]:
df.describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,...,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714,26.302232,166.950000,37.525446,...,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393,39.773434,225.715373,54.628979,...,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,...,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000



1. 구매 관련
- MntWines: 와인 구매가 평균 303으로 가장 많음 (와인이 핵심 제품일 수 있음)
- MntMeatProducts, MntFishProducts: 고기와 생선도 많이 소비됨
- 나머지는 상대적으로 덜 소비됨

2. 행동 및 캠페인
- NumWebVisitsMonth: 월 평균 웹 방문 수 약 5회 (웹사이트 방문 잦음)
- AcceptedCmp1 ~ AcceptedCmp5, ResponseL 마케팅 캠페인 수락률은 낮음 (각 7% 내외)
- Recency: 평균 49일 → 최근 방문 고객도 많고, 이탈 고객도 존재함
- Complain: 고객 불만 제기는 거의 없음 (0.9%)

### 1. 데이터 전처리 및 탐색

#### 1-1. 이상치 처리

In [84]:

# IQR 기반 이상치 탐지 함수
def detect_outliers_iqr(df, column_list):
    outlier_dict = {}
    for col in column_list:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_dict[col] = outliers
    return outlier_dict

# 이상치 컬럼 지정
column_outlines = ['Income', 'MntWines', 'MntFruits',
                   'MntMeatProducts', 'MntFishProducts',
                   'MntSweetProducts', 'MntGoldProds']

# 이상치 탐지
outlier_dict = detect_outliers_iqr(df, column_outlines)

# Plotly 시각화: Boxplot
fig = go.Figure()
for col in column_outlines:
    fig.add_trace(go.Box(
        y=df[col],
        name=col,
        boxpoints='outliers',  # outlier만 시각화
        marker=dict(size=3),
        line=dict(width=1)
    ))

fig.update_layout(
    title="📦 IQR 기반 이상치 시각화 (Boxplot) - Before Winsorization",
    yaxis_title="값 (Value)",
    xaxis_title="컬럼 (Column)",
    template="plotly_white",
    height=600
)
fig.show()


In [85]:


from scipy.stats.mstats import winsorize

def winsorize_columns(df, columns, limits=(0.01, 0.01)):
    df_winsorized = df.copy()
    for col in columns:
        df_winsorized[col] = winsorize(df[col], limits=limits)
    return df_winsorized

# 적용
df_winsorized = winsorize_columns(df, column_outlines)


In [86]:


# 이상치 컬럼 지정
column_outlines = ['Income', 'MntWines', 'MntFruits',
                   'MntMeatProducts', 'MntFishProducts',
                   'MntSweetProducts', 'MntGoldProds']

# 이상치 탐지
outlier_dict = detect_outliers_iqr(df_winsorized, column_outlines)

# Plotly 시각화: Boxplot
fig = go.Figure()
for col in column_outlines:
    fig.add_trace(go.Box(
        y=df_winsorized[col],
        name=col,
        boxpoints='outliers',  # outlier만 시각화
        marker=dict(size=3),
        line=dict(width=1)
    ))

fig.update_layout(
    title="📦 IQR 기반 이상치 시각화 (Boxplot) - After Winsorization",
    yaxis_title="값 (Value)",
    xaxis_title="컬럼 (Column)",
    template="plotly_white",
    height=600
)
fig.show()


### 2. RFM (Recency, Frequency, Monetary) 분석 변수 생성

1. R
- 이미 Recency 컬럼으로 존재

2. F
- 고객의 총 구매 횟수 (온라인/매장/카탈로그 등 합산)

3. M
- 고객의 총 구매 금액 (모든 제품군 구매금액의 합계)

In [87]:
# Recency는 이미 있음: 'Recency' 컬럼 사용

# Frequency: 다양한 구매 채널 합산
df['Frequency'] = (
    df['NumWebPurchases'] +
    df['NumCatalogPurchases'] +
    df['NumStorePurchases']
)

# Monetary: 전체 지출 금액
df['Monetary'] = (
    df['MntWines'] +
    df['MntFruits'] +
    df['MntMeatProducts'] +
    df['MntFishProducts'] +
    df['MntSweetProducts'] +
    df['MntGoldProds']
)

# 확인
df_rfm = df[['ID', 'Recency', 'Frequency', 'Monetary']]
df_rfm.head()


,ID,Recency,Frequency,Monetary
0,5524,58,22,1617
1,2174,38,4,27
2,4141,26,20,776
3,6182,26,6,53
4,5324,94,14,422


### 3. 세분화(Clustering 기반 User Segmentation)

In [88]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 스케일링 후 클러스터링
rfm_features = df_rfm[['Recency', 'Frequency', 'Monetary']]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

kmeans = KMeans(n_clusters=4, random_state=42)
df_rfm['Segment'] = kmeans.fit_predict(rfm_scaled)


In [89]:
df_rfm.head()

,ID,Recency,Frequency,Monetary,Segment
0,5524,58,22,1617,1
1,2174,38,4,27,0
2,4141,26,20,776,2
3,6182,26,6,53,0
4,5324,94,14,422,3


In [90]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)  # rfm_scaled는 표준화된 RFM 데이터
df_plot = pd.DataFrame(rfm_pca, columns=['PC1', 'PC2'])
df_plot['Segment'] = df_rfm['Segment'].astype(str)

fig = px.scatter(df_plot, x='PC1', y='PC2', color='Segment',
                 title='PCA 기반 고객 세그먼트 시각화',
                 hover_data=[df_rfm.index])

fig.show()


#### 3-2. cluster 별 score 

In [91]:
# scoring
def r_score(x): return pd.qcut(x, 5, labels=[5, 4, 3, 2, 1])  # 낮을수록 좋음
def fm_score(x): return pd.qcut(x, 5, labels=[1, 2, 3, 4, 5])  # 높을수록 좋음

df_rfm['R_score'] = r_score(df_rfm['Recency']).astype(int)
df_rfm['F_score'] = fm_score(df_rfm['Frequency']).astype(int)
df_rfm['M_score'] = fm_score(df_rfm['Monetary']).astype(int)

df_rfm['RFM_Score'] = df_rfm['R_score'].astype(int) + df_rfm['F_score'].astype(int) + df_rfm['M_score'].astype(int)

df_rfm.head()



,ID,Recency,Frequency,Monetary,Segment,R_score,F_score,M_score,RFM_Score
0,5524,58,22,1617,1,3,5,5,13
1,2174,38,4,27,0,4,1,1,6
2,4141,26,20,776,2,4,4,4,12
3,6182,26,6,53,0,4,2,1,7
4,5324,94,14,422,3,1,3,3,7


In [92]:
# 클러스터별 평균 RFM score 계산
cluster_summary = df_rfm.groupby('Segment')[['R_score', 'F_score', 'M_score']].mean().round(2)
cluster_summary


,R_score,F_score,M_score
Segment,,,
0,4.21,1.82,1.92
1,1.86,4.21,4.34
2,4.26,4.28,4.36
3,1.79,1.85,1.93


1. Cluster 0 -> 초기 관심 고객
- R: 평균 4.21 → 오랜 기간 방문 없음
- F: 평균 1.82 → 방문 빈도 낮음
- M: 평균 1.92 → 소비 금액 낮음
- 전반적으로 비활성 상태인 고객군
- 마케팅 효율이 낮을 수 있으며, 필요 시 이탈 방지용 리마케팅 고려

2. Cluster 1 -> VIP
- R: 평균 1.86 → 최근에 방문함
- F: 평균 4.21 → 매우 자주 구매
- M: 평균 4.34 → 구매 금액도 큼
- 최근 구매 이력이 있고, 자주 많은 금액을 소비하는 핵심 고객
- 매출 기여도가 높으며, 유지·우대가 필요한 고객군

3. Cluster 2 -> 이탈 가능 우량 고객
- R: 평균 4.26 → 오랜 기간 방문 없음
- F: 평균 4.28 → 자주 방문
- M: 평균 4.36 → 지출 금액도 큼
- R, F, M 모두 고점 -> 과거 VIP 였던 고객
- 자주 많이 샀던 과거 기록이 있으나, 최근엔 방문 없음

4. Cluster 3 -> 신규 유입 고객
- R: 평균 1.79 → 최근 방문
- F: 평균 1.85 → 방문 빈도 낮음
- M: 평균 1.93 → 지출 금액 낮음
- 최근 유입되었지만 아직 구매 빈도 및 금액이 낮은 초기 고객
- 활성화를 유도하기 위한 프로모션이나 리마인더 캠페인이 필요



### 4. 세그먼트 프로파일링 & 시각화

#### 4-1. segment 별 분포

In [93]:
import plotly.express as px

# Segment 분포 계산
segment_counts = df_rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']

# 막대그래프
fig = px.bar(segment_counts,
             x='Segment',
             y='Count',
             text='Count',
             title='📊 클러스터별 고객 수 분포',
             labels={'Segment': '클러스터', 'Count': '고객 수'},
             color='Segment')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()


In [94]:
# 파이차트
fig = px.pie(segment_counts,
             names='Segment',
             values='Count',
             title='📈 클러스터별 고객 비율',
             hole=0.4)
fig.update_traces(textinfo='percent+label')
fig.show()


#### 4-2. segment 별 총 구매액 및 Recency

In [95]:

# Segment별 총 Monetary 합계
monetary_by_segment = df_rfm.groupby('Segment')['Monetary'].sum().reset_index()

# 시각화
fig = px.bar(monetary_by_segment,
             x='Segment',
             y='Monetary',
             text='Monetary',
             title='💰 클러스터별 총 구매 금액',
             labels={'Segment': '클러스터', 'Monetary': '총 구매 금액'},
             color='Segment')
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()


#### 4-3. segment 별 RFM 지표 분포 비교

In [96]:
import plotly.express as px

rfm_avg = df_rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().reset_index()
rfm_avg_melted = rfm_avg.melt(id_vars='Segment', var_name='Metric', value_name='Average')

fig = px.bar(rfm_avg_melted,
             x='Segment',
             y='Average',
             color='Metric',
             barmode='group',
             title='📊 RFM 평균값 by Segment')

fig.show()


#### 4-4. segment 별 고객 비율 vs 매출 기여율


1. Cluster 0: 비활성/저가 고객군
- 많은 수의 고객이 포함되어 있으나, 매출에는 거의 기여하지 않음

2. Cluster 1: VIP 고객군
- 소수의 고객이 전체 매출의 절반 가까이를 담당

3. Cluster 2: 충성 고객군 (잠재 이탈)
- 과거 지출은 컸지만 최근 방문은 적음

4. Cluster 3: 신규/저가 고객군
-  최근 방문 고객이나 구매력이 낮음

In [97]:
import plotly.graph_objects as go

# 클러스터별 고객 수와 전체 고객 대비 비율
cluster_counts = df_rfm['Segment'].value_counts().sort_index()
customer_ratio = cluster_counts / cluster_counts.sum()

# 클러스터별 총 매출(Monetary 합계) 및 비율
monetary_by_cluster = df_rfm.groupby('Segment')['Monetary'].sum()
sales_ratio = monetary_by_cluster / monetary_by_cluster.sum()

# Plotly 이중 바 차트
fig = go.Figure()

# 고객 비율 막대
fig.add_trace(go.Bar(
    x=customer_ratio.index.astype(str),
    y=customer_ratio.values,
    name='고객 비율',
    marker_color='skyblue'
))

# 매출 기여율 막대
fig.add_trace(go.Bar(
    x=sales_ratio.index.astype(str),
    y=sales_ratio.values,
    name='매출 기여율',
    marker_color='orange'
))

# 그래프 레이아웃 설정
fig.update_layout(
    title='📊 클러스터별 고객 비율 vs 매출 기여율',
    xaxis_title='클러스터',
    yaxis_title='비율',
    barmode='group',
    yaxis_tickformat='.0%',
    legend=dict(title='지표'),
    template='plotly_white'
)

fig.show()


#### 4-5. 인구통계학적 프로파일링

1. Cluster 0: 초기 관심 고객
- 평균 나이: 54.2세 — 비교적 젊은 편
- 평균 자녀 수: 1.20명 — 자녀 수가 많은 편
- 자녀가 있는 중년 고객군으로, 아직 구매 활동은 적지만 향후 전환 가능성이 있는 잠재 고객

2. Cluster 1: VIP 고객
- 평균 나이: 58.8세 — 전체 클러스터 중 최고
- 평균 자녀 수: 0.61명 — 자녀 수는 적음
- 자녀 부담이 줄어든 고소득 중장년층 고객. 재정 여유가 있고 구매 빈도 및 금액 모두 높음

3. Cluster 2: 충성 고객
- 평균 나이: 57.4세 — 중장년층
- 평균 자녀 수: 0.59명 — 적은 편
- VIP와 유사한 성향을 보이나 최근 구매는 드문 편. 과거 높은 지출 이력 보유

4. Cluster 3: 이탈 가능 고객
- 평균 나이: 55.1세
- 평균 자녀 수: 1.26명 — 전체에서 가장 많음
- 최근 유입되었지만 구매력이 낮고 자녀 부담이 높은 고객군. 이탈 방지를 위한 맞춤 마케팅 필요



In [98]:
df_full = df.merge(df_rfm[['ID', 'Segment']], on='ID')

In [99]:
df_full.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Frequency,Monetary,Segment
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,0,0,0,0,3,11,1,22,1617,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,0,0,0,0,3,11,0,4,27,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,0,0,0,0,3,11,0,20,776,2
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,0,0,0,0,3,11,0,6,53,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,0,0,0,0,3,11,0,14,422,3


In [100]:
# 나이 및 자녀 수 파생 변수 생성
df_full['Age'] = 2025 - df_full['Year_Birth']
df_full['Total_Children'] = df_full['Kidhome'] + df_full['Teenhome']

# 클러스터별 인구통계 평균 요약
demo_summary = df_full.groupby('Segment')[['Age', 'Total_Children']].mean().round(1)

# 클러스터별 범주형 변수 분포
edu_dist = df_full.groupby(['Segment', 'Education']).size().unstack().fillna(0)
marital_dist = df_full.groupby(['Segment', 'Marital_Status']).size().unstack().fillna(0)

In [101]:
import plotly.express as px

# 예: df_full에 Segment, Age, Total_Children 컬럼이 포함되어 있다고 가정
fig_age = px.bar(
    df_full.groupby('Segment')['Age'].mean().reset_index(),
    x='Segment',
    y='Age',
    text='Age',
    title='📊 클러스터별 평균 나이',
    labels={'Segment': '클러스터', 'Age': '평균 나이'},
    color='Segment'
)
fig_age.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_age.show()

fig_kids = px.bar(
    df_full.groupby('Segment')['Total_Children'].mean().reset_index(),
    x='Segment',
    y='Total_Children',
    text='Total_Children',
    title='👨‍👩‍👧‍👦 클러스터별 평균 자녀 수',
    labels={'Segment': '클러스터', 'Total_Children': '평균 자녀 수'},
    color='Segment'
)
fig_kids.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_kids.show()


#### 4-6. 클러스터별 마케팅 채널 유입

1. Cluster 0

- 평균 웹 방문 수: 6.34 (매우 높음)

- 평균 캠페인 응답 수: 0.26 (낮음)

- 특성: 관심은 있지만 실제 반응은 낮은 비활성 고객군

- 추천 채널: 리타겟팅 중심의 SNS 광고, 디스플레이 광고

2. Cluster 1

- 평균 웹 방문 수: 3.98 (중간 이하)

- 평균 캠페인 응답 수: 0.69 (높음)

- 특성: 방문 빈도는 적지만 응답률이 높은 충성 고객군

- 추천 채널: 이메일 마케팅, SMS, DM 기반의 유지 및 업셀링 캠페인

3. Cluster 2

- 평균 웹 방문 수: 4.09 (중간)

- 평균 캠페인 응답 수: 0.85 (매우 높음)

- 특성: 과거에 많이 소비했으며 캠페인 반응률도 높음 (잠재 이탈 VIP)

- 추천 채널: 프리미엄 프로모션, 개인화된 전화 마케팅, 단기 리인게이지먼트

4. Cluster 3

- 평균 웹 방문 수: 6.32 (매우 높음)

- 평균 캠페인 응답 수: 0.12 (매우 낮음)

- 특성: 웹 활동은 활발하지만 구매 전환율이 매우 낮음

- 추천 채널: 앱 푸시, 카카오 알림톡, 팝업 리마인더 등 행동 유도 중심 채널

In [102]:

# 캠페인 응답 수 합산
df_full['TotalCampaignAcceptance'] = df_full[['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']].sum(axis=1)

# 클러스터별 집계
channel_df = df_full.groupby('Segment').agg({
    'NumWebVisitsMonth': 'mean',
    'TotalCampaignAcceptance': 'mean'
}).reset_index()

# 1. 웹 방문 수
fig1 = px.bar(channel_df, x='Segment', y='NumWebVisitsMonth',
              color='Segment', text=channel_df['NumWebVisitsMonth'].round(2),
              title='📱 클러스터별 평균 웹 방문 수', labels={'NumWebVisitsMonth': '평균 웹 방문 수'})
fig1.update_layout(yaxis_tickformat='.1f')
fig1.show()

# 2. 캠페인 응답 수
fig2 = px.bar(channel_df, x='Segment', y='TotalCampaignAcceptance',
              color='Segment', text=channel_df['TotalCampaignAcceptance'].round(2),
              title='📩 클러스터별 평균 캠페인 응답 수', labels={'TotalCampaignAcceptance': '평균 응답 수'})
fig2.update_layout(yaxis_tickformat='.1f')
fig2.show()


### 5. 마케팅 인사이트 도출

고객의 RFM 점수를 기준으로 4개의 클러스터를 도출하였습니다. 각 군집은 구매 시점, 빈도, 금액의 뚜렷한 차이를 보여 실제 고객 행동과 잘 부합하며, VIP, 이탈 가능 고객, 신규 및 비활성 고객으로 구분됩니다. 이를 통해 타겟별 마케팅 전략 수립이 용이합니다.

#### 1. Cluster 0: 비활성 고객 (초기 관심 → 이탈 위험)
- Recency 높음 (오랜 기간 방문 없음)
- Frequency / Monetary 낮음
- 웹 방문 수 가장 많음
- 캠페인 반응도 낮음

 -  마케팅 전략
    - 이탈 방지 또는 관계 회복 중심
    - 강력한 할인 / 무료 혜택 제안
    - 행동 기반 리마케팅(웹 행동 추적) 우선

- 채널
    - 웹 리타겟팅 광고 (방문 이력 기반)
    - 문자 메시지 (짧고 즉각적인 CTA)
    - 이메일은 보조 채널로 활용

- 메시지 예시
    - “오랜만이에요! 돌아오시면 혜택이 기다리고 있어요”
    - “지금 재방문하면 전용 쿠폰 드려요!”

#### 2. Cluster 1: VIP 고객
- RFM 점수 모두 우수 (최근 방문, 높은 구매력)
- 고객 수는 적지만 매출 기여 1위
- 캠페인 반응률 높음
-  웹 방문 수 낮음

-  마케팅 전략
    - 유지 & 로열티 강화
    - VIP 전용 이벤트, 조기 접근 혜택
    - 정서적 유대감 형성 강화

- 채널
    - 프리미엄 이메일 (고급스러운 디자인)
    - 1:1 전용 푸시 / 맞춤 메시지
    - 오프라인 초청 행사 또는 전화 캠페인

- 메시지 예시
    - “프리미엄 고객님만을 위한 선공개 혜택!”
    - “VIP 전용 초청장 – 단 48시간 한정”


#### 3. Cluster 2: 이탈 가능 우수 고객
- 과거에 자주 많이 구매했지만 최근 활동 없음
- 캠페인 반응률 가장 높음
- 웹 방문 평균 수준

- 마케팅 전략
    - 재방문 유도 집중
    - 지난 구매 기반 개인화 추천
    - 리마인더 + 한정 이벤트 조합

- 채널
    - 이메일 / 앱 푸시 중심
    - 개인화 캠페인 (이전에 구매한 상품 기반)

- 메시지 예시
    - “지난번에 좋아하셨던 상품이 다시 입고되었습니다!”
    - “다시 방문하시면 지난 혜택이 복원돼요”


#### 4. Cluster 3: 신규 유입 고객
- 최근 유입되었지만 F/M 낮음 (초기 유저)
- 웹 방문 수 많음
- 캠페인 반응 매우 낮음

- 마케팅 전략
    - 첫 구매 전환 유도
    - 온보딩 콘텐츠 강화
    - UX 기반의 초기 리텐션 설계

- 채널
    - 인앱 메시지, 푸시 알림
    - 웹팝업 + 첫 구매 전용 쿠폰
    - 자동화된 온보딩 이메일 시리즈

- 메시지 예시
    - “첫 구매 혜택 놓치지 마세요!”
    - “신규 고객님을 위한 전용 웰컴 쿠폰”

### 6. 제출용

In [103]:
# 클러스터 숫자 라벨에 이름 부여
segment_name_map = {
    0: '비활성 고객',     # 방문도 드물고 지출도 낮은 그룹
    1: 'VIP 고객',        # 최근 방문, 자주, 많이 지출
    2: '이탈 가능 우수 고객',  # 과거 지출 많았지만 최근 비활성
    3: '신규 유입 고객'    # 최근 방문했지만 아직 지출 적음
}

# 매핑된 이름으로 새 컬럼 생성
df_rfm['segmentation'] = df_rfm['Segment'].map(segment_name_map)

# 확인
df_rfm[['ID', 'Segment', 'segmentation']].head()


,ID,Segment,segmentation
0,5524,1,VIP 고객
1,2174,0,비활성 고객
2,4141,2,이탈 가능 우수 고객
3,6182,0,비활성 고객
4,5324,3,신규 유입 고객


In [104]:
# CSV 파일로 저장
df_rfm.to_csv('rfm_segmented.csv', index=False)


In [105]:
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)